<a href="https://colab.research.google.com/github/MacUpr/ML/blob/main/csv_patient.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import pandas as pd
import requests
import json

def filter_csv_with_llm(
    csv_path: str,
    filter_column: str,
    filter_value: str,
    llm_endpoint: str = "http://84.237.50.69:8848",
    llm_model_id: str = "qwen3.5:latest",
    match_type: str = "contains" # 'exact', 'contains', 'fuzzy' (requires LLM processing)
) -> pd.DataFrame:
    """
    Filters a CSV file based on a specified column and value,
    potentially leveraging an LLM for advanced matching or query understanding.

    Args:
        csv_path (str): The path to the CSV file.
        filter_column (str): The name of the column to filter (e.g., 'patient_id', 'patient_name').
        filter_value (str): The value to filter by.
        llm_endpoint (str): The URL of the LLM API endpoint.
        llm_model_id (str): The ID or model name of the LLM to use.
        match_type (str): The type of matching to perform:
                          'exact' for exact match,
                          'contains' for substring match (case-insensitive),
                          'fuzzy' for LLM-assisted fuzzy matching (requires LLM to provide results or pattern).
                          Currently, 'fuzzy' will default to 'contains' if LLM does not provide specific instructions.

    Returns:
        pd.DataFrame: A DataFrame containing the filtered rows. Returns an empty DataFrame on error.
    """
    try:
        df = pd.read_csv(csv_path)
    except FileNotFoundError:
        print(f"Error: CSV file not found at {csv_path}")
        return pd.DataFrame()
    except Exception as e:
        print(f"Error reading CSV file: {e}")
        return pd.DataFrame()

    if filter_column not in df.columns:
        print(f"Error: Filter column '{filter_column}' not found in the CSV.")
        return pd.DataFrame()

    print(f"Preparing to filter by '{filter_column}' for value '{filter_value}'.")

    # LLM Interaction (if match_type is 'fuzzy' or for guidance)
    llm_guidance = ""
    refined_filter_value = filter_value

    if match_type == "fuzzy":
        print(f"Attempting LLM-assisted fuzzy matching with LLM: {llm_model_id} at {llm_endpoint}")
        # Construct a prompt for the LLM to help with fuzzy matching.
        # This could involve asking the LLM to generate regex, suggest alternative spellings,
        # or evaluate if certain entries match.
        llm_prompt = (
            f"Given the column '{filter_column}' from a patient CSV and a search term '{filter_value}', "
            "provide potential variations, common misspellings, or a regular expression that would "
            "help in finding relevant entries. If the search term itself needs clarification, suggest it. "
            "Focus on producing a refined search term or a simple matching strategy. "
            "Respond concisely with the most relevant refined term or a simple instruction (e.g., 'use regex: ...', 'exact match: ...')."
        )

        llm_payload = {
            "model": llm_model_id,
            "messages": [
                {"role": "system", "content": "You are a helpful assistant for data filtering."},
                {"role": "user", "content": llm_prompt}
            ],
            "max_tokens": 150,
            "temperature": 0.5
        }

        try:
            # Assuming an OpenAI-compatible API endpoint for chat completions
            response = requests.post(f"{llm_endpoint}/v1/chat/completions", json=llm_payload, timeout=30)
            response.raise_for_status()
            llm_response_json = response.json()
            llm_guidance = llm_response_json.get("choices", [{}])[0].get("message", {}).get("content", "").strip()
            print(f"LLM Guidance for fuzzy match: {llm_guidance}")

            # Attempt to parse LLM guidance for a refined filter value or strategy
            # This parsing is highly dependent on how the LLM responds.
            # For simplicity, if LLM suggests an "exact match: X", we'll use X.
            if llm_guidance.lower().startswith("exact match:"):
                refined_filter_value = llm_guidance.split(":", 1)[1].strip()
                print(f"LLM suggested exact match with: '{refined_filter_value}'")
                match_type = "exact" # Override match_type for this specific query
            elif llm_guidance.lower().startswith("use regex:"):
                # If LLM provides regex, we would need to implement regex filtering
                print(f"LLM suggested regex, but for simplicity, defaulting to 'contains' for now. LLM output: {llm_guidance}")
                # For now, default to 'contains' if regex parsing is too complex or not implemented.
                match_type = "contains"
                # refined_filter_value = # Extract regex pattern if implemented
            else:
                # If LLM just gives general advice, or no clear refined term, stick to original value
                print("LLM guidance for fuzzy matching was not directly parseable into a refined term or strategy. Proceeding with 'contains'.")
                match_type = "contains"


        except requests.exceptions.RequestException as e:
            print(f"Error communicating with LLM at {llm_endpoint}: {e}")
            print("Proceeding with standard CSV filtering without LLM fuzzy assistance.")
            llm_guidance = "" # Reset guidance
            match_type = "contains" # Fallback if LLM fails for fuzzy

    # Perform actual filtering
    filtered_df = pd.DataFrame()
    column_series = df[filter_column].astype(str).str.strip()

    if match_type == "exact":
        filtered_df = df[column_series.str.lower() == refined_filter_value.lower()]
    elif match_type == "contains":
        filtered_df = df[column_series.str.contains(refined_filter_value, case=False, na=False)]
    else: # Default to contains if match_type is unrecognized or fuzzy failed
        print(f"Unrecognized or failed match_type '{match_type}'. Defaulting to 'contains'.")
        filtered_df = df[column_series.str.contains(refined_filter_value, case=False, na=False)]

    if filtered_df.empty:
        print(f"No results found for '{refined_filter_value}' in column '{filter_column}' with '{match_type}' matching.")
    else:
        print(f"Found {len(filtered_df)} matching entries using '{match_type}' matching.")

    return filtered_df

# To use this tool:
# 1. Replace '/path/to/your/patient_data.csv' with the actual path to your CSV file.
# 2. Replace 'patient_id' or 'patient_name' with the correct column name in your CSV.
# 3. Adjust 'search_value' to the patient ID or name you are looking for.
# 4. Choose 'match_type' as 'exact', 'contains', or 'fuzzy'.

# Example of how you would call the function after defining it:
#
# # Example 1: Basic 'contains' match
# # filtered_patients_contains = filter_csv_with_llm(
# #     csv_path="/content/sample_data/california_housing_test.csv", # Replace with your CSV
# #     filter_column="ocean_proximity", # Replace with 'patient_name' or 'patient_id'
# #     filter_value="ISLAND",
# #     match_type="contains"
# # )
# # print("\n--- Contains Match Results (Example 1) ---")
# # print(filtered_patients_contains.head())
#
# # Example 2: LLM-assisted 'fuzzy' match (will try to interact with LLM)
# # filtered_patients_fuzzy = filter_csv_with_llm(
# #     csv_path="/content/sample_data/california_housing_test.csv", # Replace with your CSV
# #     filter_column="ocean_proximity", # Replace with 'patient_name' or 'patient_id'
# #     filter_value="near bay", # Example for fuzzy match
# #     match_type="fuzzy"
# # )
# # print("\n--- Fuzzy Match Results (Example 2) ---")
# # print(filtered_patients_fuzzy.head())

In [3]:
!pip install -q kaggle

import os
from google.colab import userdata

try:
    os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
    os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')
except:
    print("Kaggle credentials not found in secrets. Please upload kaggle.json manually or set secrets.")

# Updated to the specific Healthcare Dataset
DATASET_NAME = 'prasad22/healthcare-dataset'
!kaggle datasets download -d {DATASET_NAME} --unzip

print("Healthcare dataset downloaded and unzipped.")

Kaggle credentials not found in secrets. Please upload kaggle.json manually or set secrets.
Dataset URL: https://www.kaggle.com/datasets/prasad22/healthcare-dataset
License(s): CC0-1.0
100% 2.91M/2.91M [00:00<00:00, 127MB/s]

Healthcare dataset downloaded and unzipped.


In [4]:
# Apply the tool to the Healthcare CSV
csv_file = 'healthcare_dataset.csv'

# Example: Filtering for a patient named 'Tiffany' using fuzzy match
# The dataset contains columns: Name, Age, Gender, Blood Type, Medical Condition, etc.
results = filter_csv_with_llm(
    csv_path=csv_file,
    filter_column='Name',
    filter_value='Tiffany',
    match_type='fuzzy'
)

print("Filtered Results:")
display(results.head())

Preparing to filter by 'Name' for value 'Tiffany'.
Attempting LLM-assisted fuzzy matching with LLM: qwen3.5:latest at http://84.237.50.69:8848
LLM Guidance for fuzzy match: 
LLM guidance for fuzzy matching was not directly parseable into a refined term or strategy. Proceeding with 'contains'.
Found 201 matching entries using 'contains' matching.
Filtered Results:


,Name,Age,Gender,Blood Type,Medical Condition,Date of Admission,Doctor,Hospital,Insurance Provider,Billing Amount,Room Number,Admission Type,Discharge Date,Medication,Test Results
197,MRs. TiFFAny DeNnIs,26,Male,O-,Diabetes,2021-10-11,James James DVM,Group Scott,UnitedHealthcare,13769.403129,498,Urgent,2021-10-24,Ibuprofen,Abnormal
436,tiFfAnY FErguSON,85,Female,A+,Cancer,2022-09-10,Eric Lutz,Johnson Group,Blue Cross,27971.280361,120,Emergency,2022-09-14,Lipitor,Abnormal
560,TiFFAny LEe,32,Female,A+,Obesity,2023-03-07,Maria Estes,Rodriguez-Arnold,UnitedHealthcare,38668.694589,462,Urgent,2023-03-12,Lipitor,Normal
1872,TifFanY cooPEr,33,Female,AB+,Cancer,2022-08-11,Alexandra Pena,Wright Sons and,Blue Cross,37153.379739,280,Urgent,2022-09-09,Aspirin,Abnormal
2324,TIfFany DukE,82,Male,O+,Arthritis,2020-08-18,Brandon Middleton,"Ross, and Chambers Webb",UnitedHealthcare,43834.147909,404,Emergency,2020-09-03,Lipitor,Inconclusive


In [5]:
# Ensure you have run the first cell in the notebook to define 'filter_csv_with_llm' before running this.
column_to_search = 'Name'
value_to_find = 'Tiffany'

try:
    final_results = filter_csv_with_llm(
        csv_path='healthcare_dataset.csv',
        filter_column=column_to_search,
        filter_value=value_to_find,
        match_type='fuzzy'
    )

    if not final_results.empty:
        print(f"\nSearch Complete. Found {len(final_results)} matching records:")
        display(final_results)
    else:
        print("No matches found. Check your search term or if the dataset was downloaded correctly.")
except NameError:
    print("ERROR: The function 'filter_csv_with_llm' is not defined. Please run the first cell of this notebook first!")

Preparing to filter by 'Name' for value 'Tiffany'.
Attempting LLM-assisted fuzzy matching with LLM: qwen3.5:latest at http://84.237.50.69:8848
LLM Guidance for fuzzy match: 
LLM guidance for fuzzy matching was not directly parseable into a refined term or strategy. Proceeding with 'contains'.
Found 201 matching entries using 'contains' matching.

Search Complete. Found 201 matching records:


,Name,Age,Gender,Blood Type,Medical Condition,Date of Admission,Doctor,Hospital,Insurance Provider,Billing Amount,Room Number,Admission Type,Discharge Date,Medication,Test Results
197,MRs. TiFFAny DeNnIs,26,Male,O-,Diabetes,2021-10-11,James James DVM,Group Scott,UnitedHealthcare,13769.403129,498,Urgent,2021-10-24,Ibuprofen,Abnormal
436,tiFfAnY FErguSON,85,Female,A+,Cancer,2022-09-10,Eric Lutz,Johnson Group,Blue Cross,27971.280361,120,Emergency,2022-09-14,Lipitor,Abnormal
560,TiFFAny LEe,32,Female,A+,Obesity,2023-03-07,Maria Estes,Rodriguez-Arnold,UnitedHealthcare,38668.694589,462,Urgent,2023-03-12,Lipitor,Normal
1872,TifFanY cooPEr,33,Female,AB+,Cancer,2022-08-11,Alexandra Pena,Wright Sons and,Blue Cross,37153.379739,280,Urgent,2022-09-09,Aspirin,Abnormal
2324,TIfFany DukE,82,Male,O+,Arthritis,2020-08-18,Brandon Middleton,"Ross, and Chambers Webb",UnitedHealthcare,43834.147909,404,Emergency,2020-09-03,Lipitor,Inconclusive
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
54722,TiFfAny gRaNT,87,Female,B-,Hypertension,2022-03-07,Jeanette Berry,"Mcneil Miller, Nicholson and",Blue Cross,36253.985061,213,Elective,2022-03-13,Ibuprofen,Abnormal
54723,tifFAny JAmeS,52,Male,AB-,Arthritis,2020-11-09,Amanda Callahan,Dickerson LLC,Medicare,7106.832698,299,Urgent,2020-12-02,Penicillin,Normal
55277,TIFfaNY JohNS,60,Female,O-,Hypertension,2022-03-12,Catherine Smith,Bryan PLC,Aetna,3034.602071,236,Emergency,2022-03-24,Aspirin,Inconclusive
55389,tiFfAnY FErguSON,82,Female,A+,Cancer,2022-09-10,Eric Lutz,Johnson Group,Blue Cross,27971.280361,120,Emergency,2022-09-14,Lipitor,Abnormal
